### Make temporary directory

In [1]:
import os, random, string

In [2]:
cwd = os.path.basename(os.getcwd())
tmpdir = os.path.join("/flash/ReiterU/tmp/", 'tomo_' + ''.join(random.choices(string.ascii_uppercase, k=12)))
print(tmpdir)
if not os.path.exists(tmpdir):
    os.makedirs(tmpdir)

/flash/ReiterU/tmp/tomo_YTEPYNCJWYOQ


### Main

In [3]:
from nptdms import TdmsFile
from nptdms import TdmsWriter, ChannelObject
import numpy as np
from matplotlib import pyplot as plt

In [4]:
f = TdmsFile.read("/bucket/ReiterU/OLdev/experiments/20241018_squid_cal520/sync/rec11.tdms")
g = f.groups()
twop = g[0].channels()[0][:]  # line0; TTL generated by 2P scope
cam = g[0].channels()[1][:]  # line1; TTL from basler camera
vis = g[0].channels()[2][:]  # line2; TTL from Psychopy
twop_re = np.flatnonzero((twop[:-1] < 0.5) & (twop[1:] > 0.5))

In [5]:
# crop TDMS

start_idx = twop_re[25500] - 20
end_idx = twop_re[31001] + 20

twop_crop = twop[start_idx:end_idx]
cam_crop = cam[start_idx:end_idx]
vis_crop = vis[start_idx:end_idx]

In [6]:
# write to a file

twop_ch = ChannelObject("TTL", "twop", twop_crop)
cam_ch = ChannelObject("TTL", "cam", cam_crop)
vis_ch = ChannelObject("TTL", "vis", vis_crop)

outfile = os.path.join(tmpdir, "rec11_cropped.tdms")
with TdmsWriter(outfile) as writer:
    writer.write_segment([twop_ch, cam_ch, vis_ch])

### Save to bucket

In [7]:
savedir = "/bucket/ReiterU/OLdev/analysis/20250416_IGL_waves/20241018_rec11/"
print(f"cp -r {tmpdir}/rec11_cropped.tdms {savedir}")

cp -r /flash/ReiterU/tmp/tomo_YTEPYNCJWYOQ/rec11_cropped.tdms /bucket/ReiterU/OLdev/analysis/20250416_IGL_waves/20241018_rec11/


In [8]:
# check
f = TdmsFile.read(outfile)
g = f.groups()
twop = g[0].channels()[0][:]  # line0; TTL generated by 2P scope
cam = g[0].channels()[1][:]  # line1; TTL from basler camera
vis = g[0].channels()[2][:]  # line2; TTL from Psychopy
twop_re = np.flatnonzero((twop[:-1] < 0.5) & (twop[1:] > 0.5))

In [9]:
twop_re.size

5502